# 04 · U-Net training & evaluation

Train a ResNet-34 U-Net on Sen1Floods11 and evaluate vs the Phase-3 classical baseline (`ndwi_yen_raw`, IoU 0.440).

Target: validation IoU ≥ 0.75 per PRD goal **G1**.

Sections:
1. Setup (Drive, imports, config)
2. Launch training
3. Load best checkpoint & evaluate on test split
4. Paired comparison vs classical winner (bootstrap CI + McNemar)
5. Qualitative 4×4 grid (chip, classical, U-Net, GT)
6. Export ONNX for portable inference

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

REPO_ROOT = Path.cwd() if (Path.cwd() / 'PRD.md').exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.data.sen1floods11_loader import Sen1Floods11Dataset
from src.models.unet import UNetConfig, build_unet, count_parameters, load_checkpoint
from src.inference.predict import predict_chip
from src.eval import metrics
from src.eval.significance import mcnemar_test, paired_bootstrap_iou, per_chip_iou
from src.eval.ablation import AblationConfig, predict as classical_predict
from src.train.train_unet import TrainConfig, train

FIG = Path('reports/figures'); FIG.mkdir(parents=True, exist_ok=True)

SEN1F_ROOT = Path(os.environ.get('SEN1FLOODS11_DIR', '/content/drive/MyDrive/dda/sen1floods11'))
CKPT_DIR   = Path(os.environ.get('DDA_CHECKPOINT_DIR', '/content/drive/MyDrive/dda/checkpoints/unet_resnet34'))
print('Sen1Floods11 :', SEN1F_ROOT, SEN1F_ROOT.exists())
print('Checkpoints  :', CKPT_DIR)
print('CUDA         :', torch.cuda.is_available())

## 2 · Launch training

Expect ~4–6 hrs on a Colab free-tier T4 at default settings. For a smoke test, drop `epochs=3` and `batch_size=4`.

**Safe to re-run** — will resume from `last.pt` if present, keep appending to `metrics.csv`, and overwrite `best.pt` only on improvement.

In [ ]:
cfg = TrainConfig(
    sen1floods11_root=str(SEN1F_ROOT),
    out_dir=str(CKPT_DIR),
    epochs=30,
    batch_size=8,
    learning_rate=1e-4,
    train_crop=256,
    val_crop=None,
    num_workers=2,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    amp=torch.cuda.is_available(),
    early_stopping_patience=8,
)
best_ckpt = train(cfg)
print('best checkpoint:', best_ckpt)

## 3 · Training curves


In [ ]:
metrics_csv = CKPT_DIR / 'metrics.csv'
dfm = pd.read_csv(metrics_csv)
dfm.tail()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(dfm['epoch'], dfm['train_loss'], label='train'); axes[0].plot(dfm['epoch'], dfm['val_loss'], label='val')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend(); axes[0].set_title('Loss')

for k in ('val_iou', 'val_f1', 'val_cohen_kappa'):
    axes[1].plot(dfm['epoch'], dfm[k], label=k)
axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].set_title('Validation metrics')
plt.tight_layout(); plt.savefig(FIG / 'phase4_training_curves.png', dpi=150, bbox_inches='tight'); plt.show()

## 4 · Evaluate on test split + compare vs classical

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_checkpoint(str(CKPT_DIR / 'best.pt'), device=device)

test_ds = Sen1Floods11Dataset(SEN1F_ROOT, split='test', modality='s2')
n = len(test_ds)

unet_preds, classical_preds, labels = [], [], []
unet_iou = np.empty(n); classical_iou = np.empty(n)
classical_cfg = AblationConfig(index='ndwi', threshold='yen', morphology=False)

from tqdm.auto import tqdm
for i in tqdm(range(n)):
    s = test_ds[i]
    img, y = s['image'].numpy(), s['label'].numpy()
    probs = predict_chip(model, img, device=device)
    p_unet = probs >= 0.5
    p_cls = classical_predict(img, classical_cfg)
    unet_iou[i] = metrics.iou(p_unet, y)
    classical_iou[i] = metrics.iou(p_cls, y)
    unet_preds.append(p_unet); classical_preds.append(p_cls); labels.append(y)

print(f'\nU-Net       IoU = {np.nanmean(unet_iou):.4f}')
print(f'classical   IoU = {np.nanmean(classical_iou):.4f}')

boot = paired_bootstrap_iou(unet_iou, classical_iou, n_bootstrap=10_000)
print(f'\nΔIoU (U-Net − classical) = {boot.mean_delta:+.4f}')
print(f'   95% CI = [{boot.ci_lower:+.4f}, {boot.ci_upper:+.4f}]')

pu = np.concatenate([p.ravel() for p in unet_preds])
pc = np.concatenate([p.ravel() for p in classical_preds])
yy = np.concatenate([y.ravel() for y in labels])
mc = mcnemar_test(pu, pc, yy)
print(f'\nMcNemar χ² = {mc.statistic:.2f}  p = {mc.p_value:.4g}  →  ' + ('SIGNIFICANT' if mc.significant() else 'not significant'))

## 5 · Qualitative 4×4 grid (classical vs U-Net vs GT)

In [ ]:
def stretch(x, lo=2, hi=98):
    a, b = np.percentile(x[np.isfinite(x)], [lo, hi])
    return np.clip((x - a) / (b - a + 1e-9), 0, 1)

rng = np.random.default_rng(42)
picks = rng.choice(n, size=4, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for row, i in enumerate(picks):
    s = test_ds[int(i)]; img = s['image'].numpy(); y = s['label'].numpy()
    rgb = np.transpose(np.stack([stretch(img[c]) for c in (2, 1, 0)]), (1, 2, 0))
    axes[row, 0].imshow(rgb); axes[row, 0].set_ylabel(s['chip_id'], fontsize=7)
    axes[row, 1].imshow(rgb); axes[row, 1].imshow(classical_preds[int(i)], cmap='Blues', alpha=0.55)
    axes[row, 2].imshow(rgb); axes[row, 2].imshow(unet_preds[int(i)], cmap='Blues', alpha=0.55)
    axes[row, 3].imshow(rgb); axes[row, 3].imshow(np.where(y == -1, 0, y), cmap='Blues', alpha=0.55)
    for ax in axes[row]: ax.set_xticks([]); ax.set_yticks([])
for ax, title in zip(axes[0], ['RGB', 'Classical (ndwi_yen_raw)', 'U-Net', 'Ground truth']):
    ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.savefig(FIG / 'phase4_qualitative_grid.png', dpi=150, bbox_inches='tight'); plt.show()

## 6 · Export ONNX (portable inference)

In [ ]:
onnx_path = CKPT_DIR / 'unet.onnx'
dummy = torch.zeros(1, 6, 512, 512, device=device)
torch.onnx.export(
    model, dummy, str(onnx_path),
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch', 2: 'H', 3: 'W'}, 'logits': {0: 'batch', 2: 'H', 3: 'W'}},
    opset_version=16,
)
print('Wrote', onnx_path, onnx_path.stat().st_size // 1e6, 'MB')

## 7 · Phase 4 exit criteria

- [ ] `best.pt` checkpoint saved in `$DDA_CHECKPOINT_DIR`.
- [ ] Training curves figure saved.
- [ ] Test-split IoU ≥ 0.75 (PRD G1).
- [ ] Paired bootstrap CI and McNemar result recorded (in `reports/phase4_report.md`).
- [ ] 4×4 qualitative grid saved.
- [ ] ONNX export successful.

Proceed → **Phase 5 · Severity & quantification**.